In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
import pandas as pd
from public_power_backend.constants import OUTPUT_DIR, PUDL_DATA_YEAR

In [ ]:
ious_df = pd.read_parquet(OUTPUT_DIR / "data_warehouse/pudl_utilities.parquet")

In [ ]:
ious_df

In [ ]:
income_df = pd.read_parquet("s3://pudl.catalyst.coop/stable/core_ferc1__yearly_income_statements_sched114.parquet", dtype_backend="pyarrow")

In [ ]:
mask = ((income_df.report_year == 2023) & (income_df.utility_type == "total") & (income_df.income_type =="net_income_loss"))

In [ ]:
income_df[mask]

In [ ]:
income_df[mask].utility_id_ferc1.is_unique

In [ ]:
ious_df = ious_df.merge(income_df[mask][["utility_id_ferc1", "dollar_value"]], how="left", on="utility_id_ferc1").rename(columns={"dollar_value": "net_income"})

Get customer counts and revenue by customer class.

* Investigate core_ferc1__yearly_sales_by_rate_schedules_sched304

In [ ]:
sales_df = pd.read_parquet("s3://pudl.catalyst.coop/stable/core_eia861__yearly_sales.parquet", dtype_backend="pyarrow")

In [ ]:
mask = ((sales_df["report_date"].dt.year == 2023) & (sales_df["business_model"] == "retail"))

In [ ]:
sales_df.service_type.value_counts()

In [ ]:
sales_df.customer_class.value_counts()

In [ ]:
# maybe state should be part of this groupby
agg_sales_df = sales_df[mask].groupby(["utility_id_eia", "customer_class"])[["customers", "sales_mwh", "sales_revenue"]].sum().reset_index()

In [ ]:
agg_sales_df.columns

In [ ]:
agg_sales_df.index.names, agg_sales_df.columns

In [ ]:
wide_df = agg_sales_df.pivot_table(
    index="utility_id_eia",
    columns="customer_class",
    values=["customers", "sales_mwh", "sales_revenue"],
    aggfunc="sum"
)

In [ ]:
wide_df

In [ ]:
wide_df.columns = [f"{cls}_{metric}" for metric, cls in wide_df.columns]

In [ ]:
wide_df = wide_df.reset_index()

In [ ]:
wide_df

In [ ]:
ious_df = ious_df.merge(wide_df, on="utility_id_eia", how="left", validate="1:1")

In [ ]:
ious_df.columns

In [ ]:
sectors = ["commercial", "industrial", "other", "residential", "transportation"]
for sector in sectors:
    ious_df[f"{sector}_revenue_per_mwh"] = ious_df[f"{sector}_sales_revenue"] / ious_df[f"{sector}_sales_mwh"]

In [ ]:
ious_df

In [ ]:
ious_df.to_csv("utility_conditions.csv")

In [ ]:
ious_df.utility_id_ferc1.isnull().value_counts()